# Neural Network Model Testing

# 1. Importing Libraries and the Data

In [153]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from tensorflow import keras
from collect_log_data import read_log_data

In [154]:
inputs, targets = read_log_data(verbose = True)

Reading prepared data from csv files...
Prepared data collected


# 2. Testing a Basic Neural Network

Below is some code involving a very basic version of the structure this neural network may end up having. It's purpose is to serve as a sanity check to make sure everything is working okay and to look for any problems that might arise.

'ReLU' is used as a hidden layer activation function since it is often one of the best performing hidden layer activation functions. 'Softmax' is used as the output activation function since it is often the best performing activation function for multiclass-classification, and this neural network design is supposed to output the probabilities of multiple classes. 'Categorical Cross-Entropy' is used as the loss function for the model because it also typically causes models doing multiclass classification to perform the best. Adam is used as the optimizer because of its (relatively) context aware dynamic learning rate, which makes it one of the best performing and most popular optimizers.

In [155]:
hidden_activation_function = 'relu'
output_activation_function = 'softmax'
loss_function = 'categorical_crossentropy'

In [156]:
nn = keras.models.Sequential(name = "Basic_Neural_Network")

nn.add(keras.layers.Input(shape = (inputs.shape[1],)))
nn.add(keras.layers.Dense(units = 15, activation = hidden_activation_function, name = "Hidden_Layer"))
nn.add(keras.layers.Dense(units = targets.shape[1], activation = output_activation_function, name = "Output_Layer"))
optimizer = keras.optimizers.Adam()

nn.compile(optimizer = optimizer, loss = loss_function)
nn.summary()

Model: "Basic_Neural_Network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Hidden_Layer (Dense)            │ (None, 15)             │           285 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 12)             │           192 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 477 (1.86 KB)

 Trainable params: 477 (1.86 KB)

 Non-trainable params: 0 (0.00 B)

In [157]:
train_inputs, test_inputs, train_targets, test_targets = train_test_split(inputs, targets, test_size = 0.1)

In [158]:
nn.fit(train_inputs, train_targets, batch_size = 32, epochs = 100, verbose = False)

In [159]:
def match_predictor_model_evaluation(test_inputs, test_targets):
	test_predictions = nn.predict(test_inputs)

	test_predictions_split = np.hsplit(test_predictions, 2)
	red_score_predictions = np.argmax(test_predictions_split[0], axis = -1)
	blu_score_predictions = np.argmax(test_predictions_split[1], axis = -1)

	expected_scores = np.hsplit(test_targets, 2)
	red_score_targets = np.argmax(expected_scores[0], axis = -1)
	blu_score_targets = np.argmax(expected_scores[1], axis = -1)

	# Original, previous evaluation
	exact_score_accuracy = sum(np.hstack((red_score_predictions, blu_score_predictions)) ==\
									np.hstack((red_score_targets, blu_score_targets))) / test_targets.shape[0] * 100

	mae = mean_absolute_error(np.hstack((red_score_targets, blu_score_targets)),\
								np.hstack((red_score_predictions, red_score_predictions)))

	correct_winners = 0
	for i in range(test_targets.shape[0]):

		if red_score_predictions[i] > blu_score_predictions[i]:
			predicted_winner = 0
		elif red_score_predictions[i] < blu_score_predictions[i]:
			predicted_winner = 1
		else:
			predicted_winner = 2 # tie
		
		if red_score_targets[i] > blu_score_targets[i]:
			actual_winner = 0
		elif red_score_targets[i] < blu_score_targets[i]:
			actual_winner = 1
		else:
			actual_winner = 2 # tie
		
		if predicted_winner == actual_winner:
			correct_winners += 1

	correct_winner_accuracy = correct_winners / test_targets.shape[0] * 100
	return exact_score_accuracy, mae, correct_winner_accuracy

In [160]:
exact_score_accuracy, mae, correct_winner_accuracy = match_predictor_model_evaluation(test_inputs, test_targets)

print(f"Exact Score Accuracy: {exact_score_accuracy:.2f}%")
print(f"MAE: {mae:.2f}")
print(f"Correct Winner Accuracy: {correct_winner_accuracy:.2f}%")

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 726us/step
Exact Score Accuracy: 38.24%
MAE: 1.43
Correct Winner Accuracy: 52.46%


The model successfully trained and predicted and seems to get a decent MAE and accuracy when guessing the correct winning team of a match. It's exact score accuracy is approximately the same as the previous design of this neural network. This accuracy is debatably acceptable since it's for whether or not it guessed the exact correct scores of both teams for a match, and 38% is arguably decently accurate for that. Especially when the MAE suggests that it's only about 1 point and a half off on average.

# 3. Testing Number of Epochs

This test is to see if increasing the number of epochs besides 100 will make any difference on the performance of the basic test model above. This will give an idea of how many epochs it will be good to use for hyperparameter testing with neuron counts and layer counts.

In [161]:
nn = keras.models.Sequential(name = "Basic_Neural_Network")

nn.add(keras.layers.Input(shape = (inputs.shape[1],)))
nn.add(keras.layers.Dense(units = 15, activation = hidden_activation_function, name = "Hidden_Layer"))
nn.add(keras.layers.Dense(units = targets.shape[1], activation = output_activation_function, name = "Output_Layer"))
optimizer = keras.optimizers.Adam()

nn.compile(optimizer = optimizer, loss = loss_function)
nn.summary()

Model: "Basic_Neural_Network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Hidden_Layer (Dense)            │ (None, 15)             │           285 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 12)             │           192 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 477 (1.86 KB)

 Trainable params: 477 (1.86 KB)

 Non-trainable params: 0 (0.00 B)

In [162]:
nn.fit(train_inputs, train_targets, batch_size = 32, epochs = 1000, verbose = False)

In [163]:
exact_score_accuracy, mae, correct_winner_accuracy = match_predictor_model_evaluation(test_inputs, test_targets)

print(f"Exact Score Accuracy: {exact_score_accuracy:.2f}%")
print(f"MAE: {mae:.2f}")
print(f"Correct Winner Accuracy: {correct_winner_accuracy:.2f}%")

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 737us/step
Exact Score Accuracy: 33.11%
MAE: 1.43
Correct Winner Accuracy: 52.25%


Since doing 1,000 epochs instead of 100 yielded approximately the same results, that means it's likely not necessary to do more than 100 epochs for testing. Next, 50 epochs will be tested to see if training time can be cut down.

In [164]:
nn = keras.models.Sequential(name = "Basic_Neural_Network")

nn.add(keras.layers.Input(shape = (inputs.shape[1],)))
nn.add(keras.layers.Dense(units = 15, activation = hidden_activation_function, name = "Hidden_Layer"))
nn.add(keras.layers.Dense(units = targets.shape[1], activation = output_activation_function, name = "Output_Layer"))
optimizer = keras.optimizers.Adam()

nn.compile(optimizer = optimizer, loss = loss_function)
nn.summary()

Model: "Basic_Neural_Network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Hidden_Layer (Dense)            │ (None, 15)             │           285 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 12)             │           192 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 477 (1.86 KB)

 Trainable params: 477 (1.86 KB)

 Non-trainable params: 0 (0.00 B)

In [165]:
nn.fit(train_inputs, train_targets, batch_size = 32, epochs = 50, verbose = False)

In [166]:
exact_score_accuracy, mae, correct_winner_accuracy = match_predictor_model_evaluation(test_inputs, test_targets)

print(f"Exact Score Accuracy: {exact_score_accuracy:.2f}%")
print(f"MAE: {mae:.2f}")
print(f"Correct Winner Accuracy: {correct_winner_accuracy:.2f}%")

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 823us/step
Exact Score Accuracy: 38.49%
MAE: 1.74
Correct Winner Accuracy: 52.25%


50 epochs also yielded approximately the same results. Next, 10 epochs will be tested.

In [167]:
nn = keras.models.Sequential(name = "Basic_Neural_Network")

nn.add(keras.layers.Input(shape = (inputs.shape[1],)))
nn.add(keras.layers.Dense(units = 15, activation = hidden_activation_function, name = "Hidden_Layer"))
nn.add(keras.layers.Dense(units = targets.shape[1], activation = output_activation_function, name = "Output_Layer"))
optimizer = keras.optimizers.Adam()

nn.compile(optimizer = optimizer, loss = loss_function)
nn.summary()

Model: "Basic_Neural_Network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Hidden_Layer (Dense)            │ (None, 15)             │           285 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 12)             │           192 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 477 (1.86 KB)

 Trainable params: 477 (1.86 KB)

 Non-trainable params: 0 (0.00 B)

In [168]:
nn.fit(train_inputs, train_targets, batch_size = 32, epochs = 10, verbose = False)

In [169]:
exact_score_accuracy, mae, correct_winner_accuracy = match_predictor_model_evaluation(test_inputs, test_targets)

print(f"Exact Score Accuracy: {exact_score_accuracy:.2f}%")
print(f"MAE: {mae:.2f}")
print(f"Correct Winner Accuracy: {correct_winner_accuracy:.2f}%")

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 766us/step
Exact Score Accuracy: 35.90%
MAE: 1.69
Correct Winner Accuracy: 48.46%


There was a slight decrease in performance with 10 epochs. 25 epochs will be tried next. It may seem like a bit much when 10 hardly had any impact on performance, but it may be necessary for models with hyperparameters that make it more complicated with more layers and hidden neurons.

In [173]:
nn = keras.models.Sequential(name = "Basic_Neural_Network")

nn.add(keras.layers.Input(shape = (inputs.shape[1],)))
nn.add(keras.layers.Dense(units = 15, activation = hidden_activation_function, name = "Hidden_Layer"))
nn.add(keras.layers.Dense(units = targets.shape[1], activation = output_activation_function, name = "Output_Layer"))
optimizer = keras.optimizers.Adam()

nn.compile(optimizer = optimizer, loss = loss_function)
nn.summary()

Model: "Basic_Neural_Network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Hidden_Layer (Dense)            │ (None, 15)             │           285 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 12)             │           192 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 477 (1.86 KB)

 Trainable params: 477 (1.86 KB)

 Non-trainable params: 0 (0.00 B)

In [174]:
nn.fit(train_inputs, train_targets, batch_size = 32, epochs = 25, verbose = False)

In [175]:
exact_score_accuracy, mae, correct_winner_accuracy = match_predictor_model_evaluation(test_inputs, test_targets)

print(f"Exact Score Accuracy: {exact_score_accuracy:.2f}%")
print(f"MAE: {mae:.2f}")
print(f"Correct Winner Accuracy: {correct_winner_accuracy:.2f}%")

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 714us/step
Exact Score Accuracy: 35.90%
MAE: 2.40
Correct Winner Accuracy: 52.25%


25 epochs still yields similar results to 50 epochs with a model of this size, so it will be kept due to the large number of hyperparameters that are going to be tested.

# 4. Testing Layer Counts and Hidden Neuron Counts

Below is code to iterate through various numbers of hidden layers and neurons in each hidden layer to test each of them in order to find the best combination. Since this test will likely take hours, the logic of the loop for this test was first tested in `hyperparameter_loop_test.py`.

In [176]:
def print_model_details(model_details):
	print("Model:")
	print(f"Hidden Layers: {model_details[0]}, Hidden Neuron Counts: {model_details[1]}")

def print_bar():
	print("----------")

min_neurons = 10
max_neurons = 20

best_exact_score = 0
best_mae = float('inf')
best_winner_accuracy = 0

best_exact_score_model = None
best_mae_model = None
best_winner_accuracy_model = None

for hidden_layer_count in range(1, 4):
	hidden_neuron_counts = [min_neurons] * hidden_layer_count
	outer_layer = 0
	print(f"-------------------- Hidden Layers: {hidden_layer_count}")
	while outer_layer < hidden_layer_count:
		nn = keras.models.Sequential(name = "Basic_Neural_Network")

		nn.add(keras.layers.Input(shape = (inputs.shape[1],)))
		for i in range(hidden_layer_count):
			nn.add(keras.layers.Dense(units = hidden_neuron_counts[i], activation = hidden_activation_function, name = f"Hidden_Layer_{i+1}"))
		nn.add(keras.layers.Dense(units = targets.shape[1], activation = output_activation_function, name = "Output_Layer"))
		optimizer = keras.optimizers.Adam()

		nn.compile(optimizer = optimizer, loss = loss_function)

		nn.fit(train_inputs, train_targets, batch_size = 32, epochs = 50, verbose = False)

		exact_score_accuracy, mae, correct_winner_accuracy = match_predictor_model_evaluation(test_inputs, test_targets)
		print(f"Hidden Neuron Counts: {hidden_neuron_counts} (Hidden Layers: {hidden_layer_count})")
		print(f"Exact Score Accuracy: {exact_score_accuracy:.2f}%")
		print(f"MAE: {mae:.2f}")
		print(f"Correct Winner Accuracy: {correct_winner_accuracy:.2f}%")

		if exact_score_accuracy > best_exact_score:
			best_exact_score = exact_score_accuracy
			best_exact_score_model = (hidden_layer_count, hidden_neuron_counts.copy())
			print_bar()
			print(f"New best Exact Score: {best_exact_score}")
			print_model_details(best_exact_score_model)

		if mae < best_mae:
			best_mae = mae
			best_mae_model = (hidden_layer_count, hidden_neuron_counts.copy())
			print_bar()
			print(f"New best MAE: {best_mae}")
			print_model_details(best_mae_model)

		if correct_winner_accuracy > best_winner_accuracy:
			best_winner_accuracy = correct_winner_accuracy
			best_winner_accuracy_model = (hidden_layer_count, hidden_neuron_counts.copy())
			print_bar()
			print(f"New best Winner Accuracy: {best_winner_accuracy}")
			print_model_details(best_winner_accuracy_model)

		current_layer = 0
		for i in range(hidden_layer_count):
			hidden_neuron_counts[i] += 1
			if hidden_neuron_counts[i] > max_neurons:
				hidden_neuron_counts[i] = min_neurons
				current_layer += 1
			else:
				break
		
		outer_layer = max(outer_layer, current_layer)

print(f"Best Exact Score: {best_exact_score}")
print_model_details(best_exact_score_model)

print(f"Best MAE: {best_mae}")
print_model_details(best_mae_model)

print(f"Best Winner Accuracy: {best_winner_accuracy}")
print_model_details(best_winner_accuracy_model)

-------------------- Hidden Layers: 1


KeyboardInterrupt: 